# Fine-tune DeBERTa-v3-base for Multi-label Aspect Classification

This notebook fine-tunes DeBERTa-v3-base as a multi-label classifier with 22 sigmoid output heads for aspect detection in telecom feedback.

## 1. Install Dependencies

In [ ]:
import os
os.environ['DATASETS_DISABLE_VIDEO'] = '1'

!pip install -q transformers datasets accelerate scikit-learn pandas

## 2. Check GPU Availability

In [ ]:
import torch

if torch.cuda.is_available():
    device = torch.device('cuda')
    print(f"✓ GPU available: {torch.cuda.get_device_name(0)}")
    print(f"  Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    device = torch.device('cpu')
    print("⚠ No GPU found. Training will be slow. Go to Runtime → Change runtime type → T4 GPU")

## 3. Upload and Load Data

Upload your `absa_dataset_cleaned.csv` file.

In [ ]:
import pandas as pd
import os

# upload file
try:
    from google.colab import files
    print("Please upload absa_dataset_cleaned.csv")
    uploaded = files.upload()
    data_path = list(uploaded.keys())[0]
except ImportError:
    # Local environment
    data_path = '../data/processed/absa_dataset_cleaned.csv'

df = pd.read_csv(data_path)
print(f"Loaded {len(df)} rows")
print(f"Columns: {list(df.columns)}")
df.head()

## 4. Prepare Multi-label Format

Group by feedback_id to create multi-label samples (one feedback can have multiple aspects).

In [ ]:
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.model_selection import train_test_split

# Group by feedback_id
feedback_df = df.groupby('feedback_id').agg({
    'feedback_text': 'first',
    'aspect': list
}).reset_index()

print(f"Unique feedback samples: {len(feedback_df)}")
print(f"Aspects per feedback (mean): {feedback_df['aspect'].apply(len).mean():.2f}")

# Multi-label binarization
mlb = MultiLabelBinarizer()
y = mlb.fit_transform(feedback_df['aspect'])

ASPECT_CLASSES = list(mlb.classes_)
NUM_LABELS = len(ASPECT_CLASSES)

print(f"\nAspect classes ({NUM_LABELS}):")
for i, aspect in enumerate(ASPECT_CLASSES):
    count = y[:, i].sum()
    print(f"  {i:2d}. {aspect}: {count} samples")

In [ ]:
# Train/val/test split
X_train_text, X_temp_text, y_train, y_temp = train_test_split(
    feedback_df['feedback_text'].tolist(), y,
    test_size=0.2, random_state=42
)

X_val_text, X_test_text, y_val, y_test = train_test_split(
    X_temp_text, y_temp,
    test_size=0.5, random_state=42
)

print(f"Train: {len(X_train_text)} | Val: {len(X_val_text)} | Test: {len(X_test_text)}")

## 5. Create Dataset and Tokenize

In [ ]:
from transformers import AutoTokenizer
from datasets import Dataset

MODEL_NAME = "microsoft/deberta-v3-base"
MAX_LENGTH = 256

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def create_dataset(texts, labels):
    return Dataset.from_dict({
        'text': texts,
        'labels': [list(map(float, label)) for label in labels]
    })

train_dataset = create_dataset(X_train_text, y_train)
val_dataset = create_dataset(X_val_text, y_val)
test_dataset = create_dataset(X_test_text, y_test)

def tokenize_function(examples):
    tokenized = tokenizer(
        examples['text'],
        padding='max_length',
        truncation=True,
        max_length=MAX_LENGTH
    )
    tokenized['labels'] = examples['labels']
    return tokenized

train_dataset = train_dataset.map(tokenize_function, batched=True, remove_columns=['text'])
val_dataset = val_dataset.map(tokenize_function, batched=True, remove_columns=['text'])
test_dataset = test_dataset.map(tokenize_function, batched=True, remove_columns=['text'])

train_dataset.set_format('torch', columns=['input_ids', 'attention_mask'], output_all_columns=True)
val_dataset.set_format('torch', columns=['input_ids', 'attention_mask'], output_all_columns=True)
test_dataset.set_format('torch', columns=['input_ids', 'attention_mask'], output_all_columns=True)

print(f"Tokenization complete.")
print(f"Sample keys: {train_dataset[0].keys()}")


## 6. Load Model with Multi-label Classification Head

In [ ]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS,
    problem_type="multi_label_classification",
    torch_dtype=torch.float32
)
model = model.to(device)
print(f"Model loaded on {device}")
print(f"Model dtype: {next(model.parameters()).dtype}")


## 7. Define Metrics

In [ ]:
import numpy as np
from sklearn.metrics import f1_score, precision_score, recall_score

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    # Apply sigmoid and threshold
    predictions = (torch.sigmoid(torch.tensor(logits)) > 0.5).numpy().astype(int)
    labels = labels.astype(int)

    return {
        'micro_f1': f1_score(labels, predictions, average='micro', zero_division=0),
        'macro_f1': f1_score(labels, predictions, average='macro', zero_division=0),
        'precision': precision_score(labels, predictions, average='micro', zero_division=0),
        'recall': recall_score(labels, predictions, average='micro', zero_division=0)
    }

## 8. Training Configuration

In [ ]:
from transformers import TrainingArguments, Trainer, DataCollatorWithPadding
import torch.nn as nn

OUTPUT_DIR = '../models/deberta_aspect_checkpoints'

# Custom collator to handle label conversion
def custom_data_collator(features):
    batch = {
        'input_ids': torch.stack([torch.tensor(f['input_ids']) for f in features]),
        'attention_mask': torch.stack([torch.tensor(f['attention_mask']) for f in features]),
        'labels': torch.tensor([f['labels'] for f in features], dtype=torch.float32)
    }
    return batch

class MultiLabelTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        loss_fn = nn.BCEWithLogitsLoss()
        loss = loss_fn(logits, labels)
        return (loss, outputs) if return_outputs else loss

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_ratio=0.1,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="micro_f1",
    greater_is_better=True,
    logging_dir='./logs',
    logging_steps=50,
    fp16=False,
    gradient_accumulation_steps=4,
    seed=42,
    report_to="none",
)

trainer = MultiLabelTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
    data_collator=custom_data_collator,
)

print("Training configuration ready.")


## 9. Train the Model

This will take ~20-30 minutes on T4 GPU.

In [ ]:
print("Starting training...")
train_result = trainer.train()

print("\n" + "="*60)
print("Training Complete!")
print("="*60)
print(f"Training time: {train_result.metrics['train_runtime']:.1f} seconds")
print(f"Final train loss: {train_result.metrics['train_loss']:.4f}")

## 10. Evaluate on Test Set

In [ ]:
print("Evaluating...")
test_results = trainer.evaluate(test_dataset)

print("\n" + "="*60)
print("Test Results")
print("="*60)
for key, value in test_results.items():
    if not key.startswith('eval_runtime'):
        print(f"{key}: {value:.4f}")

## 11. Per-class Performance Analysis

In [ ]:
from sklearn.metrics import classification_report

test_predictions = trainer.predict(test_dataset)
y_pred = (torch.sigmoid(torch.tensor(test_predictions.predictions)) > 0.5).numpy().astype(int)

print("Per-class Performance:")
print("=" * 60)
print(classification_report(y_test, y_pred, target_names=ASPECT_CLASSES, zero_division=0))

## 13. Save Model and Artifacts

In [ ]:
import json
from datetime import datetime

FINAL_MODEL_DIR = '../models/deberta_aspect'
os.makedirs(FINAL_MODEL_DIR, exist_ok=True)

# Save model and tokenizer
trainer.save_model(FINAL_MODEL_DIR)
tokenizer.save_pretrained(FINAL_MODEL_DIR)

# Save label mapping
label_mapping = {
    'id2label': {i: label for i, label in enumerate(ASPECT_CLASSES)},
    'label2id': {label: i for i, label in enumerate(ASPECT_CLASSES)},
    'classes': ASPECT_CLASSES
}

with open(f'{FINAL_MODEL_DIR}/label_mapping.json', 'w') as f:
    json.dump(label_mapping, f, indent=2)

# Save metadata
metadata = {
    'model_name': MODEL_NAME,
    'num_labels': NUM_LABELS,
    'max_length': MAX_LENGTH,
    'training_samples': len(X_train_text),
    'test_micro_f1': test_results.get('eval_micro_f1', 0),
    'test_macro_f1': test_results.get('eval_macro_f1', 0),
    'trained_at': datetime.now().isoformat(),
    'threshold': 0.5
}

with open(f'{FINAL_MODEL_DIR}/metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)

print(f"Model saved to: {FINAL_MODEL_DIR}")
print(f"Contents: {os.listdir(FINAL_MODEL_DIR)}")